<a href="https://www.kaggle.com/code/eavprog/absolute-fx-behavioral-profiling-k-means-cluste?scriptVersionId=304292176" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import pandas as pd
import numpy as np

# 1. Загрузка
file_path = '/kaggle/input/notebooks/eavprog/abscur2/abscur.csv'
df = pd.read_csv(file_path, parse_dates=['Date'])
df = df.sort_values('Date').set_index('Date')

# 2. Очистка (ML-Ready)
# Удаляем валюты, где вообще нет данных (если такие есть)
df = df.dropna(axis=1, how='all')

# Заполняем пропуски: сначала вперед (последним известным), потом назад (для самого начала истории)
df = df.ffill().bfill()

# Проверка на критические ошибки
if df.isnull().values.any():
    print("⚠️ Внимание: в данных остались пропуски!")
else:
    print("✅ Данные очищены и готовы к анализу.")

print(f"Итоговая матрица: {df.shape[0]} дней x {df.shape[1]} валют")

✅ Данные очищены и готовы к анализу.
Итоговая матрица: 5898 дней x 45 валют


## 1.1. Анализ изменчивости логарифмической доходности (Volatility)

На первом этапе подготовки данных мы рассчитываем **годовую волатильность логарифмических доходностей**. 

### Почему это важно для машинного обучения?
В отличие от анализа номинальных курсов, работа с доходностями позволяет алгоритму **K-Means** сравнивать активы разного порядка стоимости. Логарифмирование нормализует распределение, сглаживая экстремальные выбросы, что критично для корректной работы метрических алгоритмов кластеризации.

**Технические детали расчета:**
* Мы используем стандартное отклонение ежедневных логарифмических доходностей.
* Результат масштабируется к годовому значению (умножение на $\sqrt{252}$), что позволяет интерпретировать риск в привычных для финансового анализа величинах.
* **Важное отличие:** На сайте проекта представлена страница [Рейтинги волатильности](https://www.abscur.ru/p/blog-page_26.html), где используется расчет на основе *арифметической* доходности без приведения к годовому значению. Здесь же мы намеренно переходим к годовой логарифмической волатильности, чтобы подготовить данные для обучения модели.

В таблицах ниже представлены крайние значения выборки. Тикеры являются кликабельными и ведут на интерактивные графики соответствующих абсолютных курсов на сайте проекта **Abscur**.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import HTML

# Создаем базовый DataFrame для признаков (индекс — тикеры валют)
features_df = pd.DataFrame(index=df.columns)

# 1. Вычисляем ежедневные логарифмические доходности
log_returns = np.log(df / df.shift(1))

# 2. Считаем стандартное отклонение и годовое масштабирование
features_df['Volatility'] = log_returns.std() * np.sqrt(252)

# Универсальная функция для создания ссылок
def make_clickable(ticker):
    url = f"https://www.abscur.ru/p/2.html?abs={ticker}"
    return f'<a href="{url}" target="_blank">{ticker}</a>'

# Универсальная функция для вывода результатов с ПРАВИЛЬНОЙ сортировкой
def display_sorted_results(df_input, column, ascending=True, title=""):
    # Сначала сортируем ЧИСЛА, берем 5 строк, и делаем копию
    sorted_df = df_input[[column]].sort_values(by=column, ascending=ascending).head(5).copy()
    
    # Только теперь преобразуем число в красивую строку с процентами
    sorted_df[column] = (sorted_df[column] * 100).round(2).astype(str) + '%'
    
    # Делаем ссылки в индексе
    sorted_df.index = [make_clickable(t) for t in sorted_df.index]
    
    print(title)
    display(HTML(sorted_df.to_html(escape=False)))

# Вывод результатов
display_sorted_results(features_df, 'Volatility', ascending=True, 
                       title="Топ-5 валют с МИНИМАЛЬНОЙ изменчивостью доходности:")

display_sorted_results(features_df, 'Volatility', ascending=False, 
                       title="Топ-5 валют с МАКСИМАЛЬНОЙ изменчивостью доходности:")

Топ-5 валют с МИНИМАЛЬНОЙ изменчивостью доходности:


,Volatility
HKD,2.23%
SGD,2.51%
ILS,2.86%
KWD,3.11%
USD,3.17%


Топ-5 валют с МАКСИМАЛЬНОЙ изменчивостью доходности:


,Volatility
ARS,19.95%
UAH,16.08%
EGP,15.45%
RUB,12.09%
TRY,11.38%


## 1.2. Расчет среднегодового темпа роста (CAGR)

Второй ключевой признак для нашей модели — **CAGR (Compound Annual Growth Rate)**. Это геометрическая средняя доходность, которая показывает темп роста стоимости валюты, как если бы она росла с постоянной скоростью в течение всего периода исследования.

### Зачем CAGR нужен модели?
В отличие от простой средней доходности, CAGR учитывает эффект накопления (сложный процент). Это позволяет алгоритму **K-Means** более точно разделять активы на:
1. **«Стабильно растущие»** (валюты с положительным CAGR, увеличивающие свою покупательную способность).
2. **«Девальвационные»** (валюты, демонстрирующие системное снижение ценности относительно мировой корзины).

**Методология расчета:**
* Формула: $(P_{end} / P_{start})^{1/years} - 1$.
* Данные рассчитываются на основе всего доступного исторического диапазона (с 2006 года).
* Итоговое значение приводится к годовому формату в процентах, что делает его сопоставимым с волатильностью.

In [3]:
# --- РАСЧЕТ CAGR ---

# 1. Считаем временной интервал в годах
total_days = (df.index.max() - df.index.min()).days
years = total_days / 365.25

# 2. Вычисляем CAGR и сохраняем в основной DataFrame признаков
features_df['CAGR'] = (df.iloc[-1] / df.iloc[0])**(1/years) - 1

# 3. Функция для форматирования и вывода (сортируем ЧИСЛА, а не текст)
def display_sorted_results(df_input, column, ascending=False, title=""):
    # Сортируем исходные числа
    sorted_df = df_input[[column]].sort_values(by=column, ascending=ascending).head(5).copy()
    
    # Только после сортировки преобразуем в текст для красоты
    sorted_df[column] = (sorted_df[column] * 100).round(2).astype(str) + '%'
    
    # Добавляем ссылки на сайт
    sorted_df.index = [make_clickable(t) for t in sorted_df.index]
    
    print(title)
    display(HTML(sorted_df.to_html(escape=False)))

print(f"Анализируемый период: {years:.1f} лет\n")

# Вывод лидеров роста
display_sorted_results(features_df, 'CAGR', ascending=False, 
                       title="Топ-5 валют с максимальным темпом роста (CAGR):")

# Вывод лидеров падения (девальвации)
display_sorted_results(features_df, 'CAGR', ascending=True, 
                       title="Топ-5 валют с максимальным темпом снижения (девальвация):")

Анализируемый период: 19.6 лет

Топ-5 валют с максимальным темпом роста (CAGR):


,CAGR
CHF,3.06%
CNY,2.91%
ILS,2.56%
CZK,2.44%
THB,2.31%


Топ-5 валют с максимальным темпом снижения (девальвация):


,CAGR
ARS,-21.16%
TRY,-12.12%
EGP,-7.62%
UAH,-3.39%
PKR,-2.86%
